# BUS-BRA blank-image controls: Gemma 3 4B + MedGemma 4B

**20 actual VLM calls in total:** 2 models × (5 white + 5 black) × one fixed prompt. The same uniform **512 × 512 RGB** image is reused for all five trials of each color. The models use the original notebook's **exact checkpoints/revisions, BF16 loading, native processor, unchanged BUS-BRA prompt, 32-token limit, strict parser, and greedy decoding**.

**Important:** This notebook **does not rerun inference on any original ultrasound images**. It reads/verifies the frozen original 400-case dataset to preserve labels and paired IDs, but sends **only synthetic images** to both models.

**Run on Kaggle:** attach the same verified BUS-BRA dataset containing `bus_data.csv` and `Images`; enable **T4 ×2**, Internet, and the `HF_TOKEN` Kaggle secret with Google Gemma/MedGemma checkpoint access. Select **Run All**. Both models are loaded sequentially as in the original notebook.

**Outputs** are written only inside `/kaggle/working/blank_image_white_black_controls_gemma_medgemma/`, leaving the original experiment's CSVs untouched. `control_trials.csv` stores all 20 actual responses. If all five raw and parsed answers agree for a given model/color, the notebook creates **400 derived case-level predictions** (not additional VLM inferences), with the **same CSV columns and filenames** as the original. `control_summary.csv` and provenance JSON explain what was derived. If an identical-input run is unstable, the notebook **does not expand** that condition's prediction; see trial logs.

Repeats test output stability, not sample-level accuracy. Blank images are out-of-distribution and this control does not by itself establish clinical visual understanding.


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, gc, hashlib, io, os, random, re, sys, tempfile

DATA_ROOT = None  # Auto-find bus_data.csv; set its parent folder if multiple copies are attached.
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
SEED, FINAL_SIZE, MAX_NEW_TOKENS = 42, 400, 32
CODE_VERSION = "bus-bra-gemma-medgemma-blank-dual-control-v1"
INSTALL_TIMEOUT_SECONDS = 900
HF_TOKEN = None

def log(*parts):
    message = " ".join(map(str, parts))
    if HF_TOKEN:
        message = message.replace(HF_TOKEN, "[REDACTED]")
    message = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", message)
    print(f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] {message}", flush=True)

log("Notebook started:", CODE_VERSION, "| 2 models x 2 colors x 5 inferences | BF16")

PACKAGE_PINS = {'transformers': '4.57.1', 'accelerate': '1.11.0', 'bitsandbytes': '0.48.1', 'huggingface-hub': '0.36.0', 'tokenizers': '0.22.1', 'safetensors': '0.6.2', 'sentencepiece': '0.2.1'}
BUS_BRA_PROMPT = 'You are a medical assistant. Examine this breast ultrasound image.\nClassify the lesion as exactly ONE of the following:\n- benign: non-cancerous lesion\n- malignant: cancerous lesion\n\nRespond with ONLY the label: benign or malignant.'
METADATA_HASHES = {'f218d0f10b0aeeb14ff4b2cfed8631b2f4a7cf608e564548b35bddd55f3aafb3': 'original', '1a7756e34fa27cda4faa48f41c1b5a5443e2bd0af8aa2a41e24ad8b1a96ede71': 'kaggle_orvile'}

# (output tag, checkpoint, exact model/processor revision), in execution order.
MODELS = [
    ('gemma3_4b', 'google/gemma-3-4b-it', '093f9f388b31de276ce2de164bdc2081324b9767'),
    ('medgemma_4b', 'google/medgemma-4b-it', '290cda5eeccbee130f987c4ad74a59ae6f196408'),
]

# Keep ALL control outputs separate from the original /kaggle/working/*.csv files.
CONTROL_ROOT = OUTPUT_ROOT / "blank_image_white_black_controls_gemma_medgemma"
CONTROL_COLORS = {"white": (255, 255, 255), "black": (0, 0, 0)}
CONTROL_IMAGE_SIZE = (512, 512)
TRIALS_PER_COLOR = 5


## 2. Dependencies

Preserve Kaggle torch, torchvision and Pillow. Dependency versions are the same as the original Gemma/MedGemma notebook.


In [ ]:
import importlib, importlib.metadata as im
import signal, subprocess, threading
import PIL

preserved = {name: im.version(name) for name in ("torch", "torchvision", "Pillow")}
if PIL.__version__ != preserved["Pillow"]:
    raise RuntimeError("Loaded Pillow differs from installed Pillow; start a fresh Kaggle session")
for name in ("transformers", "tokenizers"):
    if name in sys.modules and im.version(name) != PACKAGE_PINS[name]:
        raise RuntimeError(f"{name} is already loaded at a different version; start a fresh session")
log("Installing pinned dependencies; preserving:", preserved)

def install_packages(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, start_new_session=True)
    def stream_output():
        for line in process.stdout:
            log("pip |", line.rstrip())
    reader = threading.Thread(target=stream_output, daemon=True)
    reader.start()
    try:
        code = process.wait(timeout=INSTALL_TIMEOUT_SECONDS)
        if code:
            raise RuntimeError(f"Dependency installation failed, exit code {code}")
    finally:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=5)
        reader.join(timeout=5)

with tempfile.TemporaryDirectory(prefix="bus_bra_install_") as folder:
    constraints = Path(folder) / "constraints.txt"
    constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n")
    install_packages([sys.executable, "-u", "-m", "pip", "install", "--no-input",
                      "--disable-pip-version-check", "--progress-bar", "off",
                      "--timeout", "60", "--retries", "2", "-c", str(constraints),
                      *[f"{k}=={v}" for k, v in PACKAGE_PINS.items()]])
importlib.invalidate_caches()
for name, version in {**PACKAGE_PINS, **preserved}.items():
    if im.version(name) != version:
        raise RuntimeError(f"Unexpected installed version: {name}")
if PIL.__version__ != preserved["Pillow"]:
    raise RuntimeError("Pillow changed in memory; start a fresh session")
log("Dependencies ready:", PACKAGE_PINS)


## 3. Frozen image IDs and dataset verification

The same 400 Cases and verified dataset hashes as the original are used **only** to build paired ground-truth rows. Reading real image files here is validation, not inference.


In [ ]:
FROZEN_IMAGE_IDS = """
bus_0001-l bus_0003-r bus_0007-l bus_0008-r bus_0009-r bus_0010-r bus_0011-l bus_0012-r
bus_0017-s bus_0029-s bus_0030-s bus_0034-r bus_0037-s bus_0041-l bus_0042-r bus_0043-s
bus_0045-r bus_0048-l bus_0049-r bus_0050-l bus_0052-r bus_0053-l bus_0054-s bus_0057-l
bus_0058-r bus_0059-l bus_0076-r bus_0082-s bus_0084-l bus_0088-s bus_0092-r bus_0095-s
bus_0098-l bus_0099-r bus_0100-l bus_0104-s bus_0105-r bus_0106-r bus_0109-s bus_0114-s
bus_0118-l bus_0120-l bus_0122-s bus_0124-r bus_0130-r bus_0133-l bus_0135-r bus_0136-l
bus_0141-r bus_0143-r bus_0145-r bus_0146-r bus_0147-r bus_0148-l bus_0149-r bus_0156-r
bus_0157-l bus_0163-r bus_0165-l bus_0166-r bus_0171-l bus_0172-r bus_0174-l bus_0179-l
bus_0185-l bus_0186-l bus_0187-l bus_0191-l bus_0193-r bus_0197-r bus_0199-l bus_0200-s
bus_0205-l bus_0206-l bus_0208-r bus_0212-r bus_0214-l bus_0217-r bus_0218-l bus_0219-l
bus_0225-r bus_0230-r bus_0231-r bus_0232-r bus_0233-s bus_0234-l bus_0236-l bus_0239-l
bus_0240-r bus_0242-r bus_0243-r bus_0247-r bus_0249-r bus_0250-r bus_0260-l bus_0262-l
bus_0263-l bus_0264-r bus_0265-s bus_0268-r bus_0269-r bus_0274-r bus_0277-l bus_0281-l
bus_0282-r bus_0284-r bus_0286-r bus_0287-l bus_0288-r bus_0289-l bus_0290-l bus_0291-r
bus_0293-s bus_0301-r bus_0303-r bus_0310-l bus_0311-r bus_0319-l bus_0320-l bus_0321-s
bus_0324-s bus_0326-r bus_0328-r bus_0329-l bus_0331-s bus_0333-s bus_0334-l bus_0335-l
bus_0336-s bus_0340-l bus_0342-r bus_0344-r bus_0345-l bus_0351-l bus_0353-l bus_0360-l
bus_0364-r bus_0366-l bus_0368-l bus_0370-r bus_0371-l bus_0373-r bus_0374-l bus_0375-r
bus_0376-l bus_0379-l bus_0381-l bus_0384-r bus_0385-l bus_0387-r bus_0389-r bus_0390-r
bus_0392-r bus_0394-r bus_0395-l bus_0396-l bus_0397-l bus_0398-r bus_0399-r bus_0400-r
bus_0405-l bus_0410-l bus_0411-l bus_0413-l bus_0414-r bus_0415-l bus_0416-l bus_0418-r
bus_0421-r bus_0422-r bus_0426-l bus_0427-l bus_0431-r bus_0432-l bus_0438-r bus_0440-r
bus_0442-r bus_0445-r bus_0446-r bus_0448-l bus_0449-r bus_0453-l bus_0454-r bus_0455-l
bus_0456-l bus_0457-l bus_0458-l bus_0464-r bus_0465-l bus_0466-l bus_0471-l bus_0472-l
bus_0475-l bus_0484-r bus_0486-r bus_0488-l bus_0493-r bus_0497-l bus_0502-r bus_0507-l
bus_0508-l bus_0509-r bus_0513-r bus_0518-l bus_0521-l bus_0522-l bus_0531-r bus_0532-r
bus_0543-l bus_0544-l bus_0546-l bus_0549-l bus_0550-l bus_0551-r bus_0555-l bus_0560-r
bus_0566-l bus_0568-r bus_0569-l bus_0571-l bus_0574-r bus_0576-r bus_0579-l bus_0581-r
bus_0582-l bus_0590-l bus_0594-r bus_0595-r bus_0596-l bus_0600-r bus_0601-r bus_0602-l
bus_0603-r bus_0604-l bus_0605-l bus_0609-l bus_0612-r bus_0620-l bus_0626-r bus_0628-l
bus_0630-l bus_0631-l bus_0633-l bus_0639-r bus_0641-l bus_0644-r bus_0645-r bus_0647-l
bus_0657-r bus_0659-r bus_0661-r bus_0662-l bus_0663-r bus_0666-r bus_0667-r bus_0669-l
bus_0672-l bus_0677-s bus_0683-s bus_0685-l bus_0687-s bus_0697-r bus_0700-s bus_0705-s
bus_0706-s bus_0707-s bus_0709-s bus_0711-s bus_0712-s bus_0713-r bus_0721-s bus_0723-s
bus_0726-s bus_0728-s bus_0730-s bus_0731-s bus_0742-s bus_0745-s bus_0747-s bus_0748-s
bus_0752-s bus_0758-s bus_0760-s bus_0765-s bus_0766-s bus_0767-s bus_0768-s bus_0770-s
bus_0772-s bus_0791-s bus_0793-s bus_0797-s bus_0801-s bus_0803-s bus_0807-s bus_0809-s
bus_0811-s bus_0812-s bus_0816-s bus_0818-s bus_0820-s bus_0825-s bus_0826-s bus_0829-s
bus_0831-s bus_0836-s bus_0838-s bus_0841-s bus_0842-s bus_0846-s bus_0850-l bus_0854-s
bus_0858-l bus_0859-r bus_0860-l bus_0861-l bus_0862-r bus_0866-r bus_0867-r bus_0868-r
bus_0869-r bus_0874-l bus_0879-r bus_0881-l bus_0882-r bus_0886-s bus_0893-l bus_0896-r
bus_0899-s bus_0906-l bus_0911-l bus_0915-l bus_0917-s bus_0918-l bus_0927-l bus_0928-r
bus_0933-l bus_0934-l bus_0935-r bus_0937-l bus_0938-l bus_0941-r bus_0943-s bus_0944-r
bus_0945-r bus_0947-l bus_0948-r bus_0951-r bus_0952-l bus_0954-l bus_0957-s bus_0960-l
bus_0962-l bus_0964-r bus_0966-l bus_0971-r bus_0972-s bus_0973-l bus_0977-l bus_0979-r
bus_0980-r bus_0981-l bus_0982-l bus_0983-l bus_0985-l bus_0986-r bus_0988-s bus_0989-l
bus_0991-s bus_0993-r bus_0994-r bus_0997-l bus_0998-s bus_1002-l bus_1003-r bus_1007-l
bus_1008-r bus_1012-l bus_1013-r bus_1014-l bus_1016-l bus_1017-l bus_1023-r bus_1024-r
bus_1027-l bus_1028-r bus_1032-l bus_1033-l bus_1035-l bus_1037-r bus_1038-l bus_1042-l
bus_1046-l bus_1052-l bus_1055-l bus_1056-r bus_1058-l bus_1060-s bus_1061-r bus_1063-r
""".split()
PILOT_CASES = ['1004', '1049', '184', '196', '228', '253', '257', '31', '314', '369', '383', '412', '450', '462', '577', '610', '800', '901', '909', '961']
EXCLUDED_CASES = ['15', '178', '300', '735', '77', '78', '856', '857', '912', '931', '970', '976', '990']
EXPECTED_MANIFEST_SHA256 = 'a8caa6ad39310c2871e53bc4b9f80d4b76c1b6f850b1c474c0abae8db1181283'


In [ ]:
from PIL import Image

search_root = Path(DATA_ROOT) if DATA_ROOT else INPUT_ROOT
metadata_files = sorted(search_root.rglob("bus_data.csv"))
if len(metadata_files) != 1:
    raise ValueError("Attach an extracted BUS-BRA dataset with Images and bus_data.csv; set DATA_ROOT if ambiguous")
SOURCE_ROOT = metadata_files[0].parent
metadata_sha = hashlib.sha256(metadata_files[0].read_bytes()).hexdigest()
if metadata_sha not in METADATA_HASHES:
    raise ValueError("BUS-BRA metadata does not match either verified dataset copy")
with metadata_files[0].open(newline="", encoding="utf-8-sig") as stream:
    metadata = list(csv.DictReader(stream))
by_id = {r["ID"]: r for r in metadata}
source_index = {r["ID"]: i for i, r in enumerate(metadata)}
if len(metadata) != 1875 or len(by_id) != len(metadata):
    raise ValueError("Unexpected metadata count or duplicate image IDs")
image_dirs = [p for p in SOURCE_ROOT.iterdir() if p.is_dir() and p.name.lower() == "images"]
if len(image_dirs) != 1:
    raise ValueError("Expected one Images folder beside bus_data.csv")
image_paths = defaultdict(list)
for path in image_dirs[0].rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
        image_paths[path.stem].append(path)

log("Loading the frozen final sample and verifying image content")
SAMPLE = []
for image_id in FROZEN_IMAGE_IDS:
    row = by_id[image_id]
    if row["Case"] in set(PILOT_CASES) | set(EXCLUDED_CASES):
        raise ValueError("Final sample overlaps pilot or excluded Cases")
    paths = image_paths[image_id]
    if len(paths) != 1:
        raise ValueError(f"Missing or ambiguous image: {image_id}")
    path = paths[0]
    SAMPLE.append({"dataset": "bus_bra", "sample_id": image_id, "image_id": image_id,
                   "image_path": path.relative_to(SOURCE_ROOT).as_posix(), "group_id": row["Case"],
                   "split": "test", "task_type": "classification", "question": BUS_BRA_PROMPT,
                   "options_json": "{}", "reference_answer": row["Pathology"].strip().lower(),
                   "category": "", "image_sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
if (len(SAMPLE) != FINAL_SIZE or len({r["sample_id"] for r in SAMPLE}) != FINAL_SIZE
        or len({r["group_id"] for r in SAMPLE}) != FINAL_SIZE
        or Counter(r["reference_answer"] for r in SAMPLE) != {"benign": 200, "malignant": 200}):
    raise ValueError("Expected 400 distinct images/Cases with 200 images per class")
# Match the original frozen manifest's exact CSV serialization without exporting another file.
manifest = io.StringIO(newline="")
writer = csv.DictWriter(manifest, fieldnames=list(SAMPLE[0]))
writer.writeheader()
writer.writerows(SAMPLE)
if hashlib.sha256(manifest.getvalue().encode()).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Frozen sample mismatch: image content, labels, Cases, paths or prompt changed")
log("Sample ready: 400 images / 400 source Cases; 200 benign + 200 malignant")
log("Dataset:", METADATA_HASHES[metadata_sha], "| manifest SHA256:", EXPECTED_MANIFEST_SHA256)
log("Fixed prompt:", repr(BUS_BRA_PROMPT))


## 4. Original strict output parser

Exactly the same parser as the real-image experiment; unparseable final answers are scored as `unparsed`.


In [ ]:
def decode_answer_tokens(tokenizer, tokens, eos_ids):
    """Remove only configured terminal EOS IDs from the parsing copy; retain raw text."""
    eos_ids = eos_ids if isinstance(eos_ids, (list, tuple)) else [eos_ids]
    eos_set = {token for token in eos_ids if token is not None}
    answer_tokens = list(tokens)
    removed = []
    while answer_tokens and answer_tokens[-1] in eos_set:
        removed.insert(0, answer_tokens.pop())
    kwargs = dict(clean_up_tokenization_spaces=False)
    return {
        "raw_output": tokenizer.decode(tokens, skip_special_tokens=True, **kwargs),
        "raw_output_with_special_tokens": tokenizer.decode(tokens, skip_special_tokens=False, **kwargs),
        "answer_text": tokenizer.decode(answer_tokens, skip_special_tokens=True, **kwargs),
        "terminal_eos_token_ids_removed": removed,
    }

def parse_answer(text):
    """Strict, deterministic final-answer parser; no label search inside reasoning."""
    response = text.strip()
    if not response:
        return None, "empty"
    # Exactly one balanced reasoning block may precede a final answer.
    if response.startswith("<think>"):
        if response.count("<think>") != 1 or response.count("</think>") != 1:
            return None, "incomplete_or_multiple_think_blocks"
        response = response.split("</think>", 1)[1].strip()
    if "<think>" in response or "</think>" in response:
        return None, "misplaced_think_tag"
    if "<answer>" in response or "</answer>" in response:
        match = re.fullmatch(r"<answer>\s*(benign|malignant)\s*</answer>", response, re.IGNORECASE)
        if match:
            return match.group(1).lower(), "answer_tag"
        return None, "malformed_or_ambiguous_answer_tag"
    match = re.fullmatch(r"(?:final answer:\s*)?(benign|malignant)", response, re.IGNORECASE)
    if match:
        return match.group(1).lower(), "direct_or_final_marker"
    return None, "no_unambiguous_final_answer"


## 5. Original model loading and generation settings

The only inference change is passing a PIL synthetic blank image object instead of opening the real ultrasound file. The prompt and decoding remain unchanged.


In [ ]:
import torch
from transformers import (AutoProcessor, Gemma3ForConditionalGeneration, GenerationConfig,
                          LogitsProcessor, LogitsProcessorList)

if not torch.cuda.is_available():
    raise RuntimeError("Enable T4 x2 in Kaggle, then start a fresh session")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    log("HF_TOKEN unavailable; public models may work, but Google checkpoints require access")
log("Runtime:", sys.version.split()[0], "| torch:", torch.__version__, "| CUDA:", torch.version.cuda)
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        log("GPU", i, torch.cuda.get_device_name(i), "| native BF16:",
            torch.cuda.is_bf16_supported(including_emulation=False))

# Preserve the v7 invalid-score rule without keeping per-token diagnostic records.
class NumericsGuard(LogitsProcessor):
    def __init__(self):
        self.invalid = False
    def __call__(self, input_ids, scores):
        self.invalid |= bool((torch.isnan(scores) | torch.isposinf(scores)).any().item()
                             or torch.isneginf(scores).all(dim=-1).any().item())
        return scores

def prepare_checked_generation(model, tokenizer):
    # Explicit kwargs outrank checkpoint defaults in Transformers 4.57.1.
    # Disabling model-default substitution also protects values equal to global defaults.
    controls = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
                "num_beams": 1, "num_return_sequences": 1, "use_cache": True,
                "temperature": 1.0, "top_k": 50, "top_p": 1.0}
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    pad = tokenizer.pad_token_id
    if pad is None:
        pad = eos[0] if isinstance(eos, (list, tuple)) else eos
    requested = GenerationConfig(**controls, eos_token_id=eos, pad_token_id=pad,
                                 bos_token_id=tokenizer.bos_token_id)
    # Inspect the installed library's actual resolution path, not just the caller object.
    resolved, leftovers = model._prepare_generation_config(
        requested, use_model_defaults=False, **controls)
    mismatches = {k: {"expected": v, "resolved": getattr(resolved, k, None)}
                  for k, v in controls.items() if getattr(resolved, k, None) != v}
    if mismatches or leftovers:
        raise RuntimeError(f"Generation settings were not enforced: {mismatches}; unused={leftovers}")
    if resolved.eos_token_id != eos or resolved.pad_token_id != pad:
        raise RuntimeError("Generation resolution changed explicit EOS/pad token IDs")
    if resolved.get_generation_mode().value != "greedy_search":
        raise RuntimeError("Resolved generation mode is not greedy_search")
    return resolved, controls

def load_model(spec, cache):
    tag, model_id, revision = spec
    log("Loading processor:", tag, "| checkpoint:", model_id, "| revision:", revision)
    processor = AutoProcessor.from_pretrained(model_id, revision=revision, token=HF_TOKEN or False,
                                              use_fast=False, cache_dir=cache)
    budget = {i: int(torch.cuda.mem_get_info(i)[0] * 0.80) for i in range(torch.cuda.device_count())}
    log("Loading BF16 model:", tag)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id, revision=revision, token=HF_TOKEN or False, torch_dtype=torch.bfloat16,
        attn_implementation="sdpa", device_map="auto", max_memory={**budget, "cpu": 0},
        low_cpu_mem_usage=True, cache_dir=cache).eval()
    if any(str(v) in {"cpu", "disk"} for v in model.hf_device_map.values()):
        raise RuntimeError("CPU/disk offloading is not permitted for this run")
    dtypes = Counter()
    for parameter in model.parameters():
        dtypes[str(parameter.dtype)] += parameter.numel()
    if not dtypes.get("torch.bfloat16", 0):
        raise RuntimeError("No BF16 model parameters loaded")
    generation, controls = prepare_checked_generation(model, processor.tokenizer)
    log(tag, "parameter dtypes:", dict(dtypes))
    log("Resolved generation:", {"use_model_defaults": False, "generation_mode":
        generation.get_generation_mode().value, **controls, "eos_token_id": generation.eos_token_id})
    return model, processor

def infer_one(model, processor, image):
    # Only the provided synthetic RGB image and the unchanged prompt enter the model.
    # Real ultrasound image files, sample IDs and ground-truth labels NEVER enter inference.
    if image.mode != "RGB" or image.size != CONTROL_IMAGE_SIZE:
        raise ValueError("Control image must be 512x512 RGB")
    messages = [{"role": "user", "content": [{"type": "image"},
                 {"type": "text", "text": BUS_BRA_PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device=device, dtype=torch.bfloat16) if v.is_floating_point() else v.to(device)
              for k, v in inputs.items()}
    generation, controls = prepare_checked_generation(model, processor.tokenizer)
    guard = NumericsGuard()
    with torch.inference_mode():
        output = model.generate(**inputs, generation_config=generation, use_model_defaults=False,
                                **controls, logits_processor=LogitsProcessorList([guard]))
    tokens = output[0, inputs["input_ids"].shape[1]:].detach().cpu().tolist()
    eos = generation.eos_token_id
    decoded = decode_answer_tokens(processor.tokenizer, tokens, eos)
    answer, _ = parse_answer(decoded["answer_text"])
    prediction = answer if answer is not None and not guard.invalid else "unparsed"
    eos_ids = eos if isinstance(eos, (list, tuple)) else [eos]
    stop = "eos" if tokens and tokens[-1] in eos_ids else "token_limit" if len(tokens) >= MAX_NEW_TOKENS else "other"
    return decoded["raw_output"], prediction, len(tokens), stop


## 6. Run white and black blank-image controls

The notebook runs **5 white and 5 black inference calls per model**, records each response, and expands a stable response to the original **400 matching case IDs and labels** for evaluation. The control outputs do not overwrite the original results.


In [ ]:
from datetime import datetime, timezone
import json
import time

SAMPLE_COLUMNS = ["idx", "question_id", "group"]
RESULT_COLUMNS = ["idx", "question_id", "group", "anatomy", "question",
                  "true_answer", "predicted_answer", "raw_output"]
SCORE_COLUMNS = ["group", "correct", "n", "accuracy"]
GROUP = "lesion_classification"
TRIAL_COLUMNS = ["model", "model_revision", "color", "trial", "predicted_answer",
                 "raw_output", "generated_tokens", "stop_reason", "latency_ms",
                 "visual_input_pixel_sha256", "completed_at", "status"]
SUMMARY_COLUMNS = ["model", "color", "completed_trials", "expected_trials", "stable",
                   "parsed_prediction", "raw_output", "derived_rows", "derived_accuracy",
                   "unique_raw_outputs", "unique_parsed_predictions", "model_revision", "status"]

def write_csv(path, rows, fields):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)

def save_results(tag, color, rows):
    correct = sum(r["true_answer"] == r["predicted_answer"] for r in rows)
    score = {"group": GROUP, "correct": correct, "n": len(rows),
             "accuracy": correct / len(rows) if rows else None}
    folder = CONTROL_ROOT / color
    write_csv(folder / f"vqa_results_{tag}.csv", rows, RESULT_COLUMNS)
    write_csv(folder / f"vqa_scores_{tag}.csv", [score], SCORE_COLUMNS)
    return score

# Original labels are used ONLY after generation, when scoring the control.
# These are the same frozen IDs, order, questions and labels as the real-image experiment.
BASE_SAMPLE_ROWS = [
    {"idx": source_index[s["sample_id"]], "question_id": s["sample_id"], "group": GROUP}
    for s in SAMPLE
]
if len(BASE_SAMPLE_ROWS) != FINAL_SIZE or len({r["question_id"] for r in BASE_SAMPLE_ROWS}) != FINAL_SIZE:
    raise RuntimeError("Frozen sample is incomplete/duplicated")

CONTROL_ROOT.mkdir(parents=True, exist_ok=True)
TRIAL_RECORDS = []
SUMMARY_RECORDS = []
for color in CONTROL_COLORS:
    folder = CONTROL_ROOT / color
    folder.mkdir(parents=True, exist_ok=True)
    write_csv(folder / "vqa_sample.csv", BASE_SAMPLE_ROWS, SAMPLE_COLUMNS)
    for spec in MODELS:
        tag = spec[0]
        save_results(tag, color, [])  # Clear stale results from previous runs.
        provenance = folder / f"derived_control_provenance_{tag}.json"
        if provenance.exists():
            provenance.unlink()
write_csv(CONTROL_ROOT / "control_trials.csv", TRIAL_RECORDS, TRIAL_COLUMNS)
write_csv(CONTROL_ROOT / "control_summary.csv", SUMMARY_RECORDS, SUMMARY_COLUMNS)

def derive_control_rows(prediction, raw_output):
    return [
        {"idx": source_index[s["sample_id"]], "question_id": s["sample_id"],
         "group": GROUP, "anatomy": "breast", "question": BUS_BRA_PROMPT,
         "true_answer": s["reference_answer"], "predicted_answer": prediction,
         "raw_output": raw_output}
        for s in SAMPLE
    ]

def run_model_controls(spec):
    tag, model_id, revision = spec
    model = processor = None
    try:
        # Original notebook's checkpoint, revision, precision, processor and generation
        # controls are unchanged. Only the image object passed into infer_one differs.
        with tempfile.TemporaryDirectory(prefix=f"bus_bra_blank_{tag}_") as cache:
            model, processor = load_model(spec, cache)
            for color, rgb in CONTROL_COLORS.items():
                blank = Image.new("RGB", CONTROL_IMAGE_SIZE, rgb)
                if blank.getpixel((0, 0)) != rgb or blank.getpixel((511, 511)) != rgb:
                    raise RuntimeError("Control image pixel verification failed")
                pixel_sha = hashlib.sha256(blank.tobytes()).hexdigest()
                condition_trials = []
                log("Starting", tag, color, f"{TRIALS_PER_COLOR} repetitions", "pixel SHA256:", pixel_sha)
                for trial_number in range(1, TRIALS_PER_COLOR + 1):
                    started = time.perf_counter()
                    raw, prediction, tokens, stop = infer_one(model, processor, blank)
                    latency_ms = round((time.perf_counter() - started) * 1000, 1)
                    record = {
                        "model": tag, "model_revision": revision, "color": color,
                        "trial": trial_number, "predicted_answer": prediction,
                        "raw_output": raw, "generated_tokens": tokens, "stop_reason": stop,
                        "latency_ms": latency_ms, "visual_input_pixel_sha256": pixel_sha,
                        "completed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                        "status": "completed",
                    }
                    condition_trials.append(record)
                    TRIAL_RECORDS.append(record)
                    write_csv(CONTROL_ROOT / "control_trials.csv", TRIAL_RECORDS, TRIAL_COLUMNS)
                    log(tag, color, f"{trial_number}/{TRIALS_PER_COLOR}",
                        "predicted:", prediction, "raw:", repr(raw),
                        f"tokens={tokens}", stop)
                raw_outputs = {r["raw_output"] for r in condition_trials}
                parsed_predictions = {r["predicted_answer"] for r in condition_trials}
                stable = (len(condition_trials) == TRIALS_PER_COLOR and
                          len(raw_outputs) == 1 and len(parsed_predictions) == 1)
                score = None
                derived_rows = []
                if stable:
                    prediction = condition_trials[0]["predicted_answer"]
                    raw_output = condition_trials[0]["raw_output"]
                    # These 400 rows are derived from ONE stable response, NOT 400 inferences.
                    derived_rows = derive_control_rows(prediction, raw_output)
                    score = save_results(tag, color, derived_rows)
                    provenance = {
                        "control": "constant_blank_image", "color": color,
                        "rgb": list(rgb), "size": list(CONTROL_IMAGE_SIZE),
                        "pixel_sha256": pixel_sha, "actual_inference_count": TRIALS_PER_COLOR,
                        "derived_prediction_count": len(derived_rows),
                        "prediction": prediction, "raw_output": raw_output,
                        "model_tag": tag, "checkpoint": model_id, "revision": revision,
                        "decoding": {"do_sample": False, "num_beams": 1,
                                     "max_new_tokens": MAX_NEW_TOKENS},
                        "prompt_sha256": hashlib.sha256(BUS_BRA_PROMPT.encode()).hexdigest(),
                        "frozen_manifest_sha256": EXPECTED_MANIFEST_SHA256,
                        "description": "Derived case-level scores; only 5 identical-input inferences per color/model",
                    }
                    provenance_path = (CONTROL_ROOT / color /
                                       f"derived_control_provenance_{tag}.json")
                    provenance_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
                else:
                    # Leave this condition's result CSV empty; don't claim 400 stable predictions.
                    log("WARNING:", tag, color, "CONTROL UNSTABLE — no 400-row expansion.")
                summary = {
                    "model": tag, "color": color, "completed_trials": len(condition_trials),
                    "expected_trials": TRIALS_PER_COLOR, "stable": stable,
                    "parsed_prediction": (condition_trials[0]["predicted_answer"] if stable else ""),
                    "raw_output": (condition_trials[0]["raw_output"] if stable else ""),
                    "derived_rows": len(derived_rows),
                    "derived_accuracy": (score["accuracy"] if score else ""),
                    "unique_raw_outputs": len(raw_outputs),
                    "unique_parsed_predictions": len(parsed_predictions),
                    "model_revision": revision,
                    "status": "stable" if stable else "unstable_no_expansion",
                }
                SUMMARY_RECORDS.append(summary)
                write_csv(CONTROL_ROOT / "control_summary.csv", SUMMARY_RECORDS, SUMMARY_COLUMNS)
                log("Finished:", tag, color, "stable:", stable,
                    "derived rows:", len(derived_rows), "score:", score)
    finally:
        model = processor = None
        gc.collect()
        torch.cuda.empty_cache()

# Run both models sequentially (no original ultrasound inference).
for spec in MODELS:
    run_model_controls(spec)
log("Completed actual blank-image inferences:", len(TRIAL_RECORDS),
    "expected:", len(MODELS) * len(CONTROL_COLORS) * TRIALS_PER_COLOR)


## 7. Download outputs

Each `white/` and `black/` folder has the **same five CSVs** as your original Gemma/MedGemma notebook. `control_trials.csv` contains only actual 20 inference trials; `control_summary.csv` and provenance JSON distinguish the derived 400-row outputs. For paired comparison use the original CSVs and either the white or black folder, with model tags `gemma3_4b` and `medgemma_4b`.


In [ ]:
from IPython.display import FileLink, display

expected_conditions = len(MODELS) * len(CONTROL_COLORS)
if len(SUMMARY_RECORDS) != expected_conditions:
    raise RuntimeError(f"Incomplete controls: {len(SUMMARY_RECORDS)}/{expected_conditions} model/color conditions")
for row in SUMMARY_RECORDS:
    if not row["stable"] or row["derived_rows"] != FINAL_SIZE:
        raise RuntimeError(f"Unstable/incomplete control: {row['model']} ({row['color']}). "
                           "Inspect control_trials.csv and control_summary.csv instead of using derived CSVs.")

paths = [CONTROL_ROOT / "control_trials.csv", CONTROL_ROOT / "control_summary.csv"]
for color in CONTROL_COLORS:
    paths.append(CONTROL_ROOT / color / "vqa_sample.csv")
    for spec in MODELS:
        tag = spec[0]
        paths.extend([
            CONTROL_ROOT / color / f"vqa_results_{tag}.csv",
            CONTROL_ROOT / color / f"vqa_scores_{tag}.csv",
            CONTROL_ROOT / color / f"derived_control_provenance_{tag}.json",
        ])
for path in paths:
    if path.exists():
        display(FileLink(os.path.relpath(path)))

log("COMPLETE:", len(TRIAL_RECORDS), "actual calls; 4 matched 400-row control CSVs; original CSVs unchanged")
